# Lab - The Conversation Lives in Your Code

Three tasks on one customer chat. You will watch Claude lose the thread, keep the thread
in a list, and see what that memory costs.

| Task | What you learn |
| --- | --- |
| 1 | The API remembers nothing between requests |
| 2 | The conversation is a list your code sends every turn |
| 3 | Every turn sends the whole list, so input tokens grow |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `FUELIX_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

# A short system line: who Claude is, our return rule, and short answers.
SYSTEM = (
    "You are ShopAssist, an online store's support assistant. "
    "Returns are accepted within 30 days of delivery. "
    "Answer in one or two sentences."
)

# Provided: every request's input token count is noted here, so Task 3 can read it.
TOKEN_LOG = []

---

## Task 1 - Two questions sent on their own

In [ ]:
# ============================================================
# TASK 1 - Two questions sent on their own
# ============================================================
#
# WHAT TO DO
#   Finish ask(): it sends a list of messages, finds the text
#   block in the reply and returns its text. Then the cell sends
#   the customer's two messages - each one ON ITS OWN.
#
# WHY IT MATTERS
#   Watch the second answer. The customer already said "I bought
#   a jacket 20 days ago", and Claude still asks which item.
#   Claude did not forget: the API is stateless. Every request
#   stands alone, and Claude sees only what that request
#   contains. The second request contained one sentence.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Then we wrap the request in a small function called ask".
#
# HOW TO DO IT
#   Three blanks inside ask():
#
#       system      the SYSTEM constant from the setup cell
#       messages    the messages list ask() received
#       "text"      the type of the block that holds the reply -
#                   a reply can hold more than one block, so we
#                   look for this one
# ============================================================

# BEGIN SOLUTION
def ask(messages):
    response = client.messages.create(
        model=MODEL,
        max_tokens=200,
        output_config={"effort": "low"},
        system=SYSTEM,
        messages=messages,
    )
    reply = None
    for block in response.content:
        if block.type == "text":
            reply = block.text
    print(reply)
    print("input tokens:", response.usage.input_tokens)
    TOKEN_LOG.append(response.usage.input_tokens)
    return reply
# SCAFFOLD: def ask(messages):
# SCAFFOLD:     response = client.messages.create(
# SCAFFOLD:         model=MODEL,
# SCAFFOLD:         max_tokens=200,
# SCAFFOLD:         output_config={"effort": "low"},
# SCAFFOLD:         system=...,     # the SYSTEM constant
# SCAFFOLD:         messages=...,   # the messages list ask() received
# SCAFFOLD:     )
# SCAFFOLD:     reply = None
# SCAFFOLD:     for block in response.content:
# SCAFFOLD:         if block.type == ...:   # "text" - the block that holds the reply
# SCAFFOLD:             reply = block.text
# SCAFFOLD:     print(reply)
# SCAFFOLD:     print("input tokens:", response.usage.input_tokens)
# SCAFFOLD:     TOKEN_LOG.append(response.usage.input_tokens)
# SCAFFOLD:     return reply
# END SOLUTION: replace each ... below with the value named beside it


# Provided: the two messages, each sent on its own.
first_reply = ask([{"role": "user", "content": "I bought a jacket 20 days ago."}])
print()
second_reply = ask([{"role": "user", "content": "Can I still return it?"}])

check("stateless", first_reply=first_reply, second_reply=second_reply)

---

## Task 2 - Keep the conversation in a list

In [ ]:
# ============================================================
# TASK 2 - Keep the conversation in a list
# ============================================================
#
# WHAT TO DO
#   Finish say(): it adds the customer's message to `history`,
#   sends the WHOLE list to ask(), and adds Claude's reply to
#   the list as an assistant message. Then send the same two
#   messages through say().
#
# WHY IT MATTERS
#   The memory of the conversation has to live in your code. On
#   every turn Claude receives the full conversation from the
#   start - so this time the second answer knows the jacket and
#   the days left.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So the memory of the conversation must live in our code".
#
# HOW TO DO IT
#   Three blanks. Two are roles, and there are only two:
#
#       "user"        the customer speaking
#       "assistant"   ShopAssist speaking
#
#   The third is what ask() sends: the history list itself.
# ============================================================

history = []
TOKEN_LOG.clear()   # provided: count only this conversation's turns


# BEGIN SOLUTION
def say(text):
    history.append({"role": "user", "content": text})
    reply = ask(history)
    history.append({"role": "assistant", "content": reply})
# SCAFFOLD: def say(text):
# SCAFFOLD:     history.append({"role": ..., "content": text})    # "user" or "assistant"? the customer
# SCAFFOLD:     reply = ask(...)                                  # the history list - the whole conversation
# SCAFFOLD:     history.append({"role": ..., "content": reply})   # "user" or "assistant"? ShopAssist
# END SOLUTION: replace each ... below with the value named beside it


say("I bought a jacket 20 days ago.")
print()
say("Can I still return it?")

check("history", history=history)

---

## Task 3 - Watch the input tokens grow

In [ ]:
# ============================================================
# TASK 3 - Watch the input tokens grow
# ============================================================
#
# WHAT TO DO
#   Add the customer's third question to the same conversation,
#   then work out how much bigger the last request was than the
#   first.
#
# WHY IT MATTERS
#   Every turn sends the whole history again, so each request is
#   bigger than the one before. That is the price of memory:
#   each turn costs more than the last, and a long chat can
#   reach the context window - the most text one request can
#   hold.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Look at the input tokens".
#
# HOW TO DO IT
#   Run Task 2 first - this cell continues its conversation.
#   One blank: the last turn's count minus the first.
#
#       TOKEN_LOG[-1]   the last turn's input tokens
#       TOKEN_LOG[0]    the first turn's input tokens
# ============================================================

say("And if I lost the receipt?")
print()
print("input tokens per turn:", TOKEN_LOG)

# BEGIN SOLUTION
growth = TOKEN_LOG[-1] - TOKEN_LOG[0]
# SCAFFOLD: growth = ...   # TOKEN_LOG[-1] - TOKEN_LOG[0]: the last turn minus the first
# END SOLUTION: replace the ... below with the value named beside it

print("the last request carried", growth, "more input tokens than the first")

check("tokens_grow", history=history, token_log=TOKEN_LOG, growth=growth)

---

## Done

Claude never remembered anything. The second answer only knew the jacket once your code
sent the jacket along. The rule from the lecture: keep the conversation in your code,
send it in full with every request, and watch the input tokens grow.